# 4. AI 에이전트 평가 — 동아리 모집 파이프라인

**exercise 4 | 에이전트 평가 (Evaluation)**

---

## 학습 목표
1. exercise 1~3에서 구현한 **APD_alter3 동아리 모집 파이프라인** AI 에이전트를 체계적으로 평가
2. **3가지 평가 차원** 이해 및 측정
 - **효과성 (Effectiveness)**: 출력 정확도·품질
 - **효율성 (Efficiency)**: 시간·비용·처리량
 - **인간 작업부하 (Human Workload)**: 인지적·물리적 부담
3. **3가지 평가 방법** 직접 적용
 - **파일럿 테스트 (Pilot Testing)**: 정의된 테스트 케이스로 정확도 측정
 - **시뮬레이션 (Simulation)**: 대량 처리 시 효율성 추정
 - **전문가 휴리스틱 평가 (Expert Heuristic Evaluation)**: LLM-as-Judge로 출력 품질 평가

## APD_alter3 파이프라인 & 평가 대상

```
INPUT → [Task 0: Application Collector 🟦]
 → [Task 1a: Essay Analyzer 🟧] ⇉ [Task 1b: Eligibility Checker 🟧] ⇉ [Task 1c: Portfolio Evaluator 🟧]
 → [Task 2: Selection Recommender 🟪]
 → [Task 3: Interview Scheduler 🟩]
 → [Task 4: Acceptance Recommender 🟪]
 → [Task 5: Notification Sender 🟩]
 → OUTPUT(합격자)
```

**이번 주 평가 대상 에이전트:**

| 에이전트 | APD Task | 평가 방법 | 측정 차원 |
|---------|---------|---------|---------|
| **Eligibility Checker** | Task 1b | 파일럿 테스트 | 효과성 |
| **Essay Analyzer** | Task 1a | 전문가 휴리스틱 평가 | 효과성 |
| **Notification Sender** | Task 5 | 전문가 휴리스틱 평가 | 효과성 |
| **전체 파이프라인** | Task 0~5 | 시뮬레이션 | 효율성 |
| **Alt 1 vs Alt 3 비교** | — | 인간 작업부하 계산 | Human Workload |

---

### 평가 프레임워크 매트릭스

| | 파일럿 테스트 | 시뮬레이션 | 전문가 휴리스틱 평가 |
|--|--|--|--|
| **효과성** | 정확도(%) 측정 | — | 출력 품질 점수 |
| **효율성** | 응답시간 측정 | 처리량·비용 추정 | — |
| **인간 작업부하** | — | Alt 간 비교 | 태스크 난이도 추정 |

In [1]:
# 필요한 라이브러리 설치
!pip install -q google-genai requests

from google import genai
from google.genai import types
from google.genai.types import HttpOptions
from google.colab import auth
import json, time, re, asyncio, random
from dataclasses import dataclass, field
from typing import List, Dict, Optional

# Vertex AI 인증 (Google 계정으로 로그인)
auth.authenticate_user()

PROJECT_ID = "scientific-management-494205"
LOCATION = "us-central1"

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
    http_options=HttpOptions(api_version="v1"),
)
print('✅ 환경 설정 완료')

✅ 환경 설정 완료


In [2]:
# ====================================================
# Week 1~3 에이전트 클래스 재정의 (셋업)
# - BaseAgentWorker 공통 인터페이스 상속
# - EligibilityCheckerAgent  (Week 1)
# - EssayAnalyzerAgent       (Week 2)
# - NotificationSenderAgent  (Week 3)
# ====================================================

class BaseAgentWorker:
    """APD 에이전트 워커의 공통 인터페이스"""
    def __init__(self, name: str, task_id: str):
        self.name = name
        self.task_id = task_id
    def run(self, input_text: str) -> str:
        raise NotImplementedError
    def __repr__(self):
        return f'{self.__class__.__name__}(name={self.name!r}, task={self.task_id!r})'


# ── Week 1: Eligibility Checker (Task 1b) ────────────────────────────────────
ELIGIBILITY_SYSTEM_PROMPT = """당신은 동아리 지원자의 자격을 검사하는 Eligibility Checker입니다.

자격 기준:
- 학년: 1~3학년만 지원 가능 (4학년 지원 불가)
- 전공: 공과대학 또는 자연과학대학 소속
- 이수 완료 필수 과목: '파이썬 기초', '선형대수' 중 최소 1개 이수

응답 형식 (반드시 준수):
판정: PASS 또는 FAIL
사유: [각 기준에 대한 구체적 설명]"""

class EligibilityCheckerAgent(BaseAgentWorker):
    def __init__(self):
        super().__init__('Eligibility Checker', 'Task 1b')
        self.config = types.GenerateContentConfig(
                          system_instruction=ELIGIBILITY_SYSTEM_PROMPT)
    def run(self, applicant_info: str) -> str:
        return client.models.generate_content(
            model='gemini-2.5-flash-lite', contents=applicant_info, config=self.config).text


# ── Week 2: Essay Analyzer (Task 1a) ─────────────────────────────────────────
ESSAY_ANALYZER_SYSTEM_PROMPT = """당신은 동아리 지원서 에세이를 평가하는 Essay Analyzer입니다.

평가 기준 (각 5점 만점):
1. 동기 (Motivation): 동아리 지원 동기가 명확한가?
2. 적합성 (Fit): 동아리 활동과 본인 역량이 연결되는가?
3. 기술 이해도 (Technical): AI·기술 관련 개념을 올바르게 이해하는가?

응답 형식 (반드시 준수):
동기: X/5
적합성: X/5
기술 이해도: X/5
총점: X/15
종합 의견: [2~3문장 구체적 평가]"""

class EssayAnalyzerAgent(BaseAgentWorker):
    def __init__(self):
        super().__init__('Essay Analyzer', 'Task 1a')
        self.config = types.GenerateContentConfig(
                          system_instruction=ESSAY_ANALYZER_SYSTEM_PROMPT)
    def run(self, essay_text: str) -> str:
        return client.models.generate_content(
            model='gemini-2.5-flash-lite', contents=essay_text, config=self.config).text


# ── Week 3: Notification Sender (Task 5) ─────────────────────────────────────
NOTIFICATION_SYSTEM_PROMPT = """당신은 동아리 모집 결과를 통보하는 Notification Sender입니다.
합격/불합격 이메일을 작성할 때 반드시 아래 3가지를 포함하세요:
① 결과 (합격 또는 불합격)
② 구체적인 사유 (에세이·면접 결과 등)
③ 향후 안내 (합격: 오리엔테이션 일정 | 불합격: 재지원 안내)
항상 공손하고 따뜻한 어조로 작성하세요."""

class NotificationSenderAgent(BaseAgentWorker):
    def __init__(self):
        super().__init__('Notification Sender', 'Task 5')
        self.config = types.GenerateContentConfig(
                          system_instruction=NOTIFICATION_SYSTEM_PROMPT)
    def run(self, input_text: str) -> str:
        return client.models.generate_content(
            model='gemini-2.5-flash-lite', contents=input_text, config=self.config).text


# ── 에이전트 목록 확인 ────────────────────────────────────────────────────────
all_agents = [EligibilityCheckerAgent(), EssayAnalyzerAgent(), NotificationSenderAgent()]
print('✅ Week 1~3 에이전트 로드 완료:')
for a in all_agents:
    print(f'  {a}')

✅ Week 1~3 에이전트 로드 완료:
  EligibilityCheckerAgent(name='Eligibility Checker', task='Task 1b')
  EssayAnalyzerAgent(name='Essay Analyzer', task='Task 1a')
  NotificationSenderAgent(name='Notification Sender', task='Task 5')


---

## 1. 파일럿 테스트 (Pilot Testing)

> **목적**: 실제 에이전트를 미리 정의된 테스트 케이스에 실행하여
> - **효과성**: 기대 출력과 실제 출력이 일치하는지 → 정확도(%)
> - **효율성**: 응답에 걸린 시간 측정

### 평가 대상: Eligibility Checker (Task 1b)

| 테스트 ID | 설명 | 예상 판정 |
|----------|------|---------|
| EC001 | 정상 지원자 (2학년, 공과대학, 파이썬 이수) | PASS |
| EC002 | 4학년 지원 불가 케이스 | FAIL |
| EC003 | 비공학 전공 (경영대학) 케이스 | FAIL |
| EC004 | 자연과학대학, 선형대수 이수 | PASS |
| EC005 | 공학 전공이나 이수 과목 없음 | FAIL |

In [3]:
# ── 테스트 케이스 정의 ─────────────────────────────────────────────────────────
@dataclass
class EvalCase:
    test_id: str
    input_text: str
    expected: str          # 'PASS' 또는 'FAIL'
    description: str

ELIGIBILITY_TEST_CASES = [
    EvalCase(
        test_id='EC001',
        input_text='이름: 이지은\n학년: 2학년\n전공: 컴퓨터공학과 (공과대학)\n이수 과목: 파이썬 기초',
        expected='PASS',
        description='정상 지원자 — 모든 조건 충족'
    ),
    EvalCase(
        test_id='EC002',
        input_text='이름: 박철수\n학년: 4학년\n전공: 전기공학과 (공과대학)\n이수 과목: 파이썬 기초, 선형대수',
        expected='FAIL',
        description='4학년 지원 불가 케이스'
    ),
    EvalCase(
        test_id='EC003',
        input_text='이름: 김민지\n학년: 1학년\n전공: 경영학과 (경영대학)\n이수 과목: 파이썬 기초',
        expected='FAIL',
        description='비공학 전공 (경영대학) 케이스'
    ),
    EvalCase(
        test_id='EC004',
        input_text='이름: 최수진\n학년: 3학년\n전공: 물리학과 (자연과학대학)\n이수 과목: 선형대수',
        expected='PASS',
        description='자연과학대학 정상 지원자'
    ),
    EvalCase(
        test_id='EC005',
        input_text='이름: 정동현\n학년: 2학년\n전공: 화학공학과 (공과대학)\n이수 과목: 없음',
        expected='FAIL',
        description='이수 과목 미충족 케이스'
    ),
]

print(f'📋 테스트 케이스 {len(ELIGIBILITY_TEST_CASES)}개 준비 완료')
for tc in ELIGIBILITY_TEST_CASES:
    print(f'  [{tc.test_id}] {tc.description} → 예상: {tc.expected}')

📋 테스트 케이스 5개 준비 완료
  [EC001] 정상 지원자 — 모든 조건 충족 → 예상: PASS
  [EC002] 4학년 지원 불가 케이스 → 예상: FAIL
  [EC003] 비공학 전공 (경영대학) 케이스 → 예상: FAIL
  [EC004] 자연과학대학 정상 지원자 → 예상: PASS
  [EC005] 이수 과목 미충족 케이스 → 예상: FAIL


In [4]:
# ── Eligibility Checker 파일럿 테스트 실행 ───────────────────────────────────
print('🔍 Eligibility Checker 파일럿 테스트 시작...\n')

checker = EligibilityCheckerAgent()
pilot_results = []

for tc in ELIGIBILITY_TEST_CASES:
    start = time.time()
    response = checker.run(tc.input_text)
    elapsed = time.time() - start

    # 응답에서 PASS/FAIL 추출
    upper = response.upper()
    actual = 'PASS' if ('PASS' in upper and 'FAIL' not in upper) else 'FAIL'
    correct = (actual == tc.expected)

    pilot_results.append({
        'test_id': tc.test_id,
        'description': tc.description,
        'expected': tc.expected,
        'actual': actual,
        'correct': correct,
        'response_time': elapsed,
        'response': response,
    })

    icon = '✅' if correct else '❌'
    print(f'[{tc.test_id}] {icon} {tc.description}')
    print(f'  예상: {tc.expected}  |  실제: {actual}  |  응답시간: {elapsed:.2f}초')
    print(f'  에이전트 출력: {response[:120].strip()}...\n')

# ── 효과성·효율성 집계 ────────────────────────────────────────────────────────
accuracy = sum(r['correct'] for r in pilot_results) / len(pilot_results)
avg_rt   = sum(r['response_time'] for r in pilot_results) / len(pilot_results)
passed   = sum(r['correct'] for r in pilot_results)

print('='*60)
print('📊 파일럿 테스트 결과 요약')
print('='*60)
print(f'  효과성(Effectiveness) — 정확도: {accuracy*100:.0f}%  ({passed}/{len(pilot_results)} 정답)')
print(f'  효율성(Efficiency)    — 평균 응답시간: {avg_rt:.2f}초/건')

🔍 Eligibility Checker 파일럿 테스트 시작...

[EC001] ✅ 정상 지원자 — 모든 조건 충족
  예상: PASS  |  실제: PASS  |  응답시간: 1.37초
  에이전트 출력: 판정: PASS
사유:
- 학년: 2학년으로 지원 가능합니다.
- 전공: 컴퓨터공학과는 공과대학 소속으로 지원 가능합니다.
- 이수 완료 필수 과목: '파이썬 기초' 과목을 이수하여 필수 요건을 충족합니다....

[EC002] ✅ 4학년 지원 불가 케이스
  예상: FAIL  |  실제: FAIL  |  응답시간: 0.57초
  에이전트 출력: 판정: FAIL
사유:
- 학년: 4학년은 지원할 수 없습니다. (1~3학년만 지원 가능)
- 전공: 공과대학 소속으로 지원 기준을 충족합니다.
- 이수 완료 필수 과목: '파이썬 기초', '선형대수' 두 과목 모두...

[EC003] ✅ 비공학 전공 (경영대학) 케이스
  예상: FAIL  |  실제: FAIL  |  응답시간: 0.70초
  에이전트 출력: 판정: FAIL
사유:
- 학년: 1학년 (지원 가능)
- 전공: 경영학과 (경영대학) (지원 불가 - 공과대학 또는 자연과학대학 소속만 지원 가능)
- 이수 완료 필수 과목: '파이썬 기초' (지원 가능)...

[EC004] ✅ 자연과학대학 정상 지원자
  예상: PASS  |  실제: PASS  |  응답시간: 0.68초
  에이전트 출력: 판정: PASS
사유:
- 학년: 3학년으로 지원 가능합니다.
- 전공: 물리학과(자연과학대학)로 지원 가능합니다.
- 이수 완료 필수 과목: '선형대수'를 이수하여 필수 과목 요건을 충족합니다....

[EC005] ✅ 이수 과목 미충족 케이스
  예상: FAIL  |  실제: FAIL  |  응답시간: 0.64초
  에이전트 출력: 판정: FAIL
사유:
- 학년: 2학년 (기준 충족)
- 전공: 화학공학과 (공과대학) (기준 충족)
- 이수 완료 필수 과목: '파이썬 기초', '선형대수' 중 최소 1개 이수 

---

## 2. 시뮬레이션 (Simulation)

> **목적**: 실제 운영 규모(지원자 수십~수백 명)에서 에이전트 성능이 어떻게 달라지는지 추정
>
> - **처리량 (Throughput)**: 분당 몇 건 처리 가능한가?
> - **비용 (Cost)**: 100명 처리 시 예상 API 비용은?
> - **병렬 vs 순차**: asyncio로 Task 1a/1b 병렬화 시 시간 단축 효과

### 시뮬레이션 시나리오

```
가상 지원서 20건 생성
 │
 ├─ 순차 처리 (5건 측정) → 100건·1000건으로 외삽
 │ └─ 처리량(건/분), 비용(USD) 추정
 │
 └─ 병렬 처리 (Task 1a ⇉ 1b, asyncio.gather)
 └─ 순차 대비 속도 향상(x배) 측정
```

In [5]:
# ── 가상 지원서 생성 ──────────────────────────────────────────────────────────
random.seed(42)

GRADES      = [1, 2, 3, 4]
MAJORS      = ['컴퓨터공학과 (공과대학)', '전기공학과 (공과대학)',
               '경영학과 (경영대학)', '물리학과 (자연과학대학)',
               '화학과 (자연과학대학)', '통계학과 (자연과학대학)']
COURSES_OPT = [['파이썬 기초'], ['선형대수'], ['파이썬 기초', '선형대수'], []]

def make_applicant(idx: int) -> str:
    grade   = random.choice(GRADES)
    major   = random.choice(MAJORS)
    courses = random.choice(COURSES_OPT)
    return (f'이름: 지원자{idx:02d}\n학년: {grade}학년\n전공: {major}\n'
            f'이수 과목: {", ".join(courses) if courses else "없음"}')

sim_applicants = [make_applicant(i) for i in range(1, 21)]
print(f'📦 가상 지원서 {len(sim_applicants)}건 생성 완료')
print(f'  샘플:\n{sim_applicants[0]}\n')

# ── 순차 처리 시간 측정 (5건 샘플로 100건 외삽) ─────────────────────────────────
print('⏱️  순차 처리 시뮬레이션 (5건 샘플)...')
checker_sim = EligibilityCheckerAgent()
seq_times = []

for app in sim_applicants[:5]:
    t0 = time.time()
    checker_sim.run(app)
    seq_times.append(time.time() - t0)

avg_seq = sum(seq_times) / len(seq_times)
print(f'  평균 처리 시간: {avg_seq:.2f}초/건')
print(f'  예상 20건 처리: {avg_seq * 20:.1f}초')
print(f'  예상 100건 처리: {avg_seq * 100 / 60:.1f}분')
print(f'  처리량: {60 / avg_seq:.1f}건/분')

# ── API 비용 추정 ──────────────────────────────────────────────────────────────
# Gemini 2.5 Flash 요금 (추정치; 실제 요금은 공식 문서 참조)
# 입력: ~$0.075/1M tokens, 출력: ~$0.30/1M tokens
EST_IN_TOK  = 120   # 건당 입력 토큰 추정
EST_OUT_TOK = 160   # 건당 출력 토큰 추정
COST_IN     = 0.075 / 1_000_000
COST_OUT    = 0.30  / 1_000_000
cost_per    = EST_IN_TOK * COST_IN + EST_OUT_TOK * COST_OUT

print(f'\n💰 비용 추정 (Gemini 2.5 Flash 기준):')
print(f'  건당 추정 비용: ${cost_per * 1000:.5f} mUSD')
print(f'  100명 처리 시: ${cost_per * 100:.5f} USD')
print(f'  1,000명 처리시: ${cost_per * 1000:.4f} USD')

📦 가상 지원서 20건 생성 완료
  샘플:
이름: 지원자01
학년: 1학년
전공: 컴퓨터공학과 (공과대학)
이수 과목: 파이썬 기초, 선형대수

⏱️  순차 처리 시뮬레이션 (5건 샘플)...
  평균 처리 시간: 0.70초/건
  예상 20건 처리: 14.0초
  예상 100건 처리: 1.2분
  처리량: 86.0건/분

💰 비용 추정 (Gemini 2.5 Flash 기준):
  건당 추정 비용: $0.05700 mUSD
  100명 처리 시: $0.00570 USD
  1,000명 처리시: $0.0570 USD


In [6]:
# ── 병렬 처리 vs 순차 처리 비교 (asyncio.gather) ─────────────────────────────
# Task 1a (Essay) / Task 1b (Eligibility) 병렬 실행 vs 순차 실행 시간 비교
#
# ⚠️ Colab/Jupyter는 이미 이벤트 루프가 실행 중이므로 asyncio.run() 대신
#    셀 최상단에서 await를 직접 사용합니다.

SAMPLE_APP = {
    'essay': ('저는 AI 기술에 깊은 관심을 가지고 있어 지원합니다. '
              '트랜스포머 모델 기반 자연어 처리 연구에 흥미가 있습니다.'),
    'eligibility': '이름: 이지은\n학년: 2학년\n전공: 컴퓨터공학과 (공과대학)\n이수 과목: 파이썬 기초',
}

essay_ag  = EssayAnalyzerAgent()
eligib_ag = EligibilityCheckerAgent()

# ── 순차 처리 ─────────────────────────────────────────────────────────────────
print('⏱️  순차 처리 (Task 1a → 1b):')
t_seq_start = time.time()
r_essay = essay_ag.run(SAMPLE_APP['essay'])
r_elig  = eligib_ag.run(SAMPLE_APP['eligibility'])
t_seq   = time.time() - t_seq_start
print(f'  Essay Analyzer  → {len(r_essay)}자 출력')
print(f'  Eligibility     → {r_elig[:60].strip()}...')
print(f'  순차 총 시간: {t_seq:.2f}초\n')

# ── 병렬 처리 (await 직접 사용) ────────────────────────────────────────────────
async def run_parallel(app: dict) -> dict:
    t1a = asyncio.to_thread(essay_ag.run,  app['essay'])
    t1b = asyncio.to_thread(eligib_ag.run, app['eligibility'])
    r1a, r1b = await asyncio.gather(t1a, t1b)
    return {'1a': r1a, '1b': r1b}

print('⚡ 병렬 처리 (Task 1a ⇉ 1b):')
t_par_start = time.time()
par_results = await run_parallel(SAMPLE_APP)   # Jupyter는 top-level await 지원
t_par       = time.time() - t_par_start
print(f'  완료. 병렬 총 시간: {t_par:.2f}초')

speedup = t_seq / t_par if t_par > 0 else 1
print(f'\n📊 결과 비교:')
print(f'  순차: {t_seq:.2f}초  |  병렬: {t_par:.2f}초  |  속도 향상: {speedup:.1f}x')
print(f'  → Task 1a/1b/1c를 병렬화하면 처리 시간을 약 {(1 - 1/speedup)*100:.0f}% 단축 가능')

⏱️  순차 처리 (Task 1a → 1b):
  Essay Analyzer  → 165자 출력
  Eligibility     → 판정: PASS
사유:
- 학년: 2학년은 지원 가능합니다.
- 전공: 컴퓨터공학과는 공과대학 소속으로 지원...
  순차 총 시간: 1.30초

⚡ 병렬 처리 (Task 1a ⇉ 1b):
  완료. 병렬 총 시간: 0.87초

📊 결과 비교:
  순차: 1.30초  |  병렬: 0.87초  |  속도 향상: 1.5x
  → Task 1a/1b/1c를 병렬화하면 처리 시간을 약 33% 단축 가능


---

## 3. 전문가 휴리스틱 평가 (Expert Heuristic Evaluation)

> **목적**: 정량적 정답이 없는 에이전트 출력(에세이 분석 결과, 합격 이메일 등)의 **품질**을
> 전문가 관점에서 평가

### LLM-as-Judge 패턴

```
에이전트 출력
 │
 ▼
[Judge LLM] ← 평가 루브릭 (전문가 기준)
 │
 ▼
품질 점수 + 판정 (GOOD / ACCEPTABLE / POOR) + 개선 의견
```

**교육 포인트**: Week 3 성찰형 에이전트의 자기 평가 메커니즘을
*독립적인 외부 평가자*로 분리한 구조입니다.

---

### 3.1 Essay Analyzer 출력 품질 평가

In [7]:
# ── 샘플 에세이 2편 준비 ──────────────────────────────────────────────────────
SAMPLE_ESSAYS = [
    {
        'id': 'E001', 'applicant': '이지은',
        'essay': (
            '저는 AI 기술에 깊은 관심을 가지고 귀 동아리에 지원합니다. '
            '특히 트랜스포머 모델 기반의 자연어 처리 연구에 흥미가 있습니다. '
            '지난 학기 파이썬 기초 과목에서 감성 분석 미니 프로젝트를 수행하며 '
            '데이터 전처리부터 모델 학습·평가까지 경험했습니다. '
            '동아리에서 팀 프로젝트를 통해 실전 AI 시스템 개발 역량을 키우고 싶습니다.'
        )
    },
    {
        'id': 'E002', 'applicant': '박민준',
        'essay': (
            'AI가 요즘 인기라고 해서 지원합니다. '
            '코딩을 조금 할 줄 알고 동아리 활동이 재미있을 것 같습니다. '
            '열심히 하겠습니다.'
        )
    },
]

# ── Essay Analyzer 실행 ─────────────────────────────────────────────────────
analyzer    = EssayAnalyzerAgent()
essay_outputs = {}

print('📝 Essay Analyzer 실행 중...\n')
for e in SAMPLE_ESSAYS:
    result = analyzer.run(e['essay'])
    essay_outputs[e['id']] = {'applicant': e['applicant'], 'analysis': result}
    print(f"[{e['id']}] {e['applicant']}의 에세이 분석 결과:")
    print(result[:300])
    print()

# ── LLM-as-Judge: 분석 품질 평가 ─────────────────────────────────────────────
JUDGE_ESSAY_PROMPT = """당신은 동아리 서류 심사 전문가입니다.
AI Essay Analyzer가 생성한 에세이 평가 결과를 검토하고 품질을 평가합니다.

평가 기준 (각 5점 만점):
1. 루브릭 준수: 동기/적합성/기술 이해도 3가지 항목이 모두 점수로 제시되었는가?
2. 점수 타당성: 에세이 내용에 비추어 점수가 논리적으로 적절한가?
3. 피드백 구체성: 종합 의견이 지원자가 개선할 수 있도록 구체적인가?

반드시 JSON 형식으로만 응답:
{"rubric_compliance": 점수, "score_validity": 점수, "feedback_specificity": 점수,
 "total": 세점수합계, "verdict": "GOOD 또는 ACCEPTABLE 또는 POOR",
 "comment": "한 문장 종합 평가"}"""

judge_essay_config = types.GenerateContentConfig(system_instruction=JUDGE_ESSAY_PROMPT)

print('⚖️  LLM-as-Judge 전문가 휴리스틱 평가 실행 중...\n')
print('='*60)

essay_judge_scores = []
for eid, data in essay_outputs.items():
    prompt = (f"평가 대상: {data['applicant']}의 에세이 분석 결과\n\n"
              f"[AI Essay Analyzer 출력]\n{data['analysis']}\n\n"
              "위 분석 결과의 품질을 JSON으로 평가하세요.")
    raw = client.models.generate_content(
        model='gemini-2.5-flash-lite', contents=prompt, config=judge_essay_config).text
    try:
        m  = re.search(r'\{[^{}]+\}', raw, re.DOTALL)
        sc = json.loads(m.group()) if m else {}
    except Exception:
        sc = {}

    essay_judge_scores.append({'id': eid, 'applicant': data['applicant'], **sc})
    v_icon = {'GOOD': '✅', 'ACCEPTABLE': '⚠️', 'POOR': '❌'}.get(sc.get('verdict', ''), '❓')
    print(f"[{eid}] {data['applicant']} 분석 품질:")
    print(f"  루브릭 준수: {sc.get('rubric_compliance','?')}/5 | "
          f"점수 타당성: {sc.get('score_validity','?')}/5 | "
          f"피드백 구체성: {sc.get('feedback_specificity','?')}/5")
    print(f"  총점: {sc.get('total','?')}/15 | 판정: {v_icon} {sc.get('verdict','?')}")
    print(f"  평가: {sc.get('comment', raw[:80])}\n")

avg_essay_quality = (sum(s.get('total', 0) for s in essay_judge_scores)
                     / max(len(essay_judge_scores), 1))
print(f'📊 Essay Analyzer 출력 평균 품질: {avg_essay_quality:.1f}/15')

📝 Essay Analyzer 실행 중...

[E001] 이지은의 에세이 분석 결과:
동기: 4/5
적합성: 4/5
기술 이해도: 3/5
총점: 11/15

종합 의견: AI 기술에 대한 명확한 관심과 동아리 지원 동기가 잘 드러납니다. 특히 자연어 처리 분야에 대한 구체적인 언급은 긍정적입니다. 다만, 감성 분석 프로젝트 경험이 기술 이해도를 뒷받침하기에는 다소 부족해 보이며, 트랜스포머 모델에 대한 좀 더 깊이 있는 이해도를 보여준다면 더 좋은 평가를 받을 수 있을 것입니다.

[E002] 박민준의 에세이 분석 결과:
동기: 1/5
적합성: 1/5
기술 이해도: 1/5
총점: 3/15
종합 의견: 지원 동기가 '인기'와 '재미'라는 다소 피상적인 이유에 머물러 있어 아쉽습니다. AI·기술에 대한 구체적인 관심이나 본인의 역량과 동아리 활동의 연결성에 대한 설명이 부족하여, 동아리 활동에 대한 열정과 잠재력을 파악하기 어렵습니다.

⚖️  LLM-as-Judge 전문가 휴리스틱 평가 실행 중...

[E001] 이지은 분석 품질:
  루브릭 준수: 5/5 | 점수 타당성: 5/5 | 피드백 구체성: 4/5
  총점: 14/15 | 판정: ✅ GOOD
  평가: AI Essay Analyzer는 명확한 기준에 따라 점수를 산정하고 구체적인 피드백을 제공하여 전반적으로 우수한 평가 결과를 도출했습니다.

[E002] 박민준 분석 품질:
  루브릭 준수: 5/5 | 점수 타당성: 5/5 | 피드백 구체성: 4/5
  총점: 14/15 | 판정: ✅ GOOD
  평가: AI Essay Analyzer는 제시된 평가 기준에 따라 박민준의 에세이를 체계적으로 분석했으며, 점수와 종합 의견 모두 에세이 내용과 잘 부합하여 전반적으로 높은 품질의 평가 결과를 제공했습니다.

📊 Essay Analyzer 출력 평균 품질: 14.0/15


### 3.2 Notification Sender 이메일 품질 평가

동일한 LLM-as-Judge 패턴을 Notification Sender에도 적용합니다.

**평가 기준**: ① 결과 명시 ② 구체적 사유 ③ 향후 안내 ④ 어조·가독성

In [8]:
# ── 합격/불합격 이메일 생성 ──────────────────────────────────────────────────
notif_agent = NotificationSenderAgent()

NOTIF_INPUTS = [
    {
        'id': 'N001', 'applicant': '이지은', 'result': '합격',
        'prompt': ('지원자: 이지은\n결과: 합격\n'
                   '사유: 에세이 기술 이해도 우수(13/15), 면접 논리적 사고력 탁월\n'
                   '오리엔테이션: 5월 3일(토) 오후 2시, 공학관 101호\n'
                   '위 정보를 바탕으로 합격 이메일을 작성하세요.')
    },
    {
        'id': 'N002', 'applicant': '박민준', 'result': '불합격',
        'prompt': ('지원자: 박민준\n결과: 불합격\n'
                   '사유: 에세이 동기·적합성 점수 낮음(5/15), 포트폴리오 미흡\n'
                   '재지원: 하반기 모집 예정\n'
                   '위 정보를 바탕으로 불합격 이메일을 작성하세요.')
    },
]

notif_outputs = {}
print('📧 Notification Sender 이메일 생성 중...\n')
for n in NOTIF_INPUTS:
    email = notif_agent.run(n['prompt'])
    notif_outputs[n['id']] = {'applicant': n['applicant'], 'result': n['result'], 'email': email}
    print(f"[{n['id']}] {n['applicant']} ({n['result']}) 이메일:")
    print(email[:300])
    print()

# ── LLM-as-Judge: 이메일 품질 평가 ──────────────────────────────────────────
JUDGE_EMAIL_PROMPT = """당신은 이메일 커뮤니케이션 전문가입니다.
동아리 모집 합격/불합격 이메일의 품질을 평가합니다.

평가 기준 (각 5점 만점):
1. 결과 명시: 합격/불합격이 명확히 기재되었는가?
2. 사유 구체성: 결정 이유가 구체적으로 설명되었는가?
3. 향후 안내: 오리엔테이션(합격) 또는 재지원 안내(불합격)가 포함되었는가?
4. 어조·가독성: 공손하고 따뜻한 어조이며 읽기 쉬운가?

반드시 JSON 형식으로만 응답:
{"result_clear": 점수, "reason_specific": 점수, "next_steps": 점수,
 "tone_readability": 점수, "total": 네점수합계,
 "verdict": "GOOD 또는 ACCEPTABLE 또는 POOR", "comment": "한 문장 종합 평가"}"""

email_judge_config = types.GenerateContentConfig(system_instruction=JUDGE_EMAIL_PROMPT)

print('⚖️  이메일 품질 LLM-as-Judge 평가 중...\n')
print('='*60)

notif_judge_scores = []
for nid, data in notif_outputs.items():
    prompt = (f"평가 대상: {data['applicant']} ({data['result']}) 이메일\n\n"
              f"[Notification Sender 출력]\n{data['email']}\n\n"
              "위 이메일의 품질을 JSON으로 평가하세요.")
    raw = client.models.generate_content(
        model='gemini-2.5-flash-lite', contents=prompt, config=email_judge_config).text
    try:
        m  = re.search(r'\{[^{}]+\}', raw, re.DOTALL)
        sc = json.loads(m.group()) if m else {}
    except Exception:
        sc = {}

    notif_judge_scores.append({'id': nid, 'applicant': data['applicant'], **sc})
    v_icon = {'GOOD': '✅', 'ACCEPTABLE': '⚠️', 'POOR': '❌'}.get(sc.get('verdict', ''), '❓')
    print(f"[{nid}] {data['applicant']} 이메일 품질:")
    print(f"  결과 명시: {sc.get('result_clear','?')}/5 | "
          f"사유 구체성: {sc.get('reason_specific','?')}/5 | "
          f"향후 안내: {sc.get('next_steps','?')}/5 | "
          f"어조: {sc.get('tone_readability','?')}/5")
    print(f"  총점: {sc.get('total','?')}/20 | 판정: {v_icon} {sc.get('verdict','?')}")
    print(f"  평가: {sc.get('comment', raw[:80])}\n")

avg_email_quality = (sum(s.get('total', 0) for s in notif_judge_scores)
                     / max(len(notif_judge_scores), 1))
print(f'📊 Notification Sender 이메일 평균 품질: {avg_email_quality:.1f}/20')

📧 Notification Sender 이메일 생성 중...

[N001] 이지은 (합격) 이메일:
## [동아리 이름] 신입 부원 모집 결과 안내 (합격)

**이지은 지원자님께,**

안녕하세요! [동아리 이름] 신입 부원 모집에 지원해주셔서 진심으로 감사드립니다.

기다리고 기다리던 [동아리 이름] 신입 부원 모집 결과, 이지은 지원자님께서 **합격**하셨음을 기쁜 마음으로 안내해 드립니다.

지원자님의 뛰어난 역량을 다시 한번 확인할 수 있었습니다. 특히, 제출해주신 에세이에서 보여주신 **기술에 대한 깊이 있는 이해도(13/15점)**는 저희를 매우 놀라게 했으며, 면접 시 보여주신 **탁월한 논리적 사고력** 또한 인상

[N002] 박민준 (불합격) 이메일:
## 안녕하세요, 박민준님. 동아리 지원 결과 안내드립니다.

먼저 저희 동아리에 관심을 갖고 귀한 시간을 내어 지원해주신 박민준님께 진심으로 감사드립니다.

안타깝게도 이번 동아리 모집 결과, 박민준님의 지원은 **불합격**으로 결정되었음을 알려드립니다.

지원해주신 에세이에서 동기와 적합성에 대한 평가 점수가 다소 낮게 나왔으며, 제출해주신 포트폴리오 또한 기대에 미치지 못하여 함께 고려되었습니다.

지원자님의 열정적인 모습과 잠재력을 보았기에 이번 결과를 알려드리는 저희도 매우 아쉬운 마음입니다.

하지만 박민준님의 역량과 열

⚖️  이메일 품질 LLM-as-Judge 평가 중...

[N001] 이지은 이메일 품질:
  결과 명시: 5/5 | 사유 구체성: 4/5 | 향후 안내: 5/5 | 어조: 5/5
  총점: 19/20 | 판정: ✅ GOOD
  평가: 합격/불합격 결과가 명확하며, 합격 사유가 구체적으로 제시되었고, 오리엔테이션 참석 안내 또한 상세하게 포함되어 있어 가독성이 높고 친절한 이메일입니다.

[N002] 박민준 이메일 품질:
  결과 명시: 5/5 | 사유 구체성: 3/5 | 향후 안내: 4/5 | 어조: 5/5
  총점: 17/20 | 판정: ✅ GOOD
  평가

---

## 4. 인간 작업부하 평가 (Human Workload)

> **목적**: AI 도입 수준에 따라 인간이 실제로 부담하는 **인지적·물리적 부하**가 어떻게 달라지는지 정량화

### APD Alternative별 인간 태스크 비교 (지원자 50명 기준)

| | Alt 1 (AI 최소 도입) | Alt 3 (AI 최대 도입) |
|--|--|--|
| **AI 담당** | Task 0, 1a, 1b | Task 0, 1a, 1b, 1c, 2(보조), 3(일정), 4(보조), 5 |
| **인간 담당** | Task 1c, 2, 3, 4, 5 | Task 2(검토), 3(인터뷰), 4(승인), 5(멘토) |

**작업부하 추정 지표:**
- **소요 시간 (분/건)**: 인간이 건당 직접 처리하는 시간
- **인지 난이도 (1~5)**: 판단·집중 등 인지적 노력 (5 = 매우 높음)
- **건수**: 실제 처리해야 하는 볼륨

In [9]:
# ── Alt 1 vs Alt 3 인간 작업부하 비교 ────────────────────────────────────────
APD_ALT = {
    'Alt 1 (AI 최소 도입)': {
        'description': 'AI: Task 0, 1a, 1b | 인간: Task 1c, 2, 3, 4, 5',
        'tasks': [
            {'name': 'Task 1c: 포트폴리오 직접 검토', 'min_per': 5,  'difficulty': 4, 'count': 50},
            {'name': 'Task 2:  면접 후보 수동 선정',   'min_per': 30, 'difficulty': 3, 'count': 1},
            {'name': 'Task 3a: 인터뷰 일정 조율',      'min_per': 20, 'difficulty': 2, 'count': 1},
            {'name': 'Task 3b: 인터뷰 진행 (20명)',     'min_per': 15, 'difficulty': 3, 'count': 20},
            {'name': 'Task 4:  최종 합격자 수동 결정',  'min_per': 60, 'difficulty': 5, 'count': 1},
            {'name': 'Task 5a: 결과 이메일 수동 발송',  'min_per': 2,  'difficulty': 1, 'count': 50},
            {'name': 'Task 5b: 멘토 직접 매칭',         'min_per': 5,  'difficulty': 2, 'count': 10},
        ]
    },
    'Alt 3 (AI 최대 도입)': {
        'description': 'AI: 대부분 | 인간: 핵심 의사결정 검토·승인만',
        'tasks': [
            {'name': 'Task 2:  AI 추천 검토 & 승인',   'min_per': 15, 'difficulty': 2, 'count': 1},
            {'name': 'Task 3:  인터뷰 진행 (20명)',     'min_per': 15, 'difficulty': 3, 'count': 20},
            {'name': 'Task 4:  AI 합격 추천 최종 승인', 'min_per': 10, 'difficulty': 2, 'count': 1},
            {'name': 'Task 5:  온보딩 멘토 매칭',       'min_per': 20, 'difficulty': 2, 'count': 1},
        ]
    }
}

alt_summary = {}
for alt_name, alt in APD_ALT.items():
    print(f'\n📌 {alt_name}')
    print(f'   {alt["description"]}')
    print(f'\n   {"태스크":<35} {"분/건":<8} {"난이도":<8} {"건수":<6} {"총 시간"}')
    print(f'   {"-"*68}')

    total_min = 0
    total_cog = 0
    for t in alt['tasks']:
        tot = t['min_per'] * t['count']
        total_min += tot
        total_cog += t['difficulty'] * t['count']
        print(f'   {t["name"]:<35} {t["min_per"]:<8} {t["difficulty"]}/5{"":<4} {t["count"]:<6} {tot}분')

    alt_summary[alt_name] = {
        'total_min': total_min, 'total_cog': total_cog,
        'task_count': len(alt['tasks'])
    }
    print(f'\n   ▶ 총 소요 시간: {total_min}분 ({total_min/60:.1f}시간)')
    print(f'   ▶ 인지 부하 합계: {total_cog}점')
    print(f'   ▶ 인간 담당 태스크 수: {len(alt["tasks"])}개')

t1 = alt_summary['Alt 1 (AI 최소 도입)']['total_min']
t3 = alt_summary['Alt 3 (AI 최대 도입)']['total_min']
c1 = alt_summary['Alt 1 (AI 최소 도입)']['total_cog']
c3 = alt_summary['Alt 3 (AI 최대 도입)']['total_cog']

print('\n' + '='*68)
print('📊 인간 작업부하 비교 요약:')
print(f'  총 소요 시간:   Alt1 {t1}분 → Alt3 {t3}분  ({(1-t3/t1)*100:.0f}% 감소 ↓)')
print(f'  인지 부하 합계: Alt1 {c1}점 → Alt3 {c3}점  ({(1-c3/c1)*100:.0f}% 감소 ↓)')
print('\n💡 AI 도입으로 인간 부하가 크게 줄었지만,')
print('   AI 오류 감지 책임과 에이전트 구축·유지 비용은 증가함을 고려해야 합니다.')


📌 Alt 1 (AI 최소 도입)
   AI: Task 0, 1a, 1b | 인간: Task 1c, 2, 3, 4, 5

   태스크                                 분/건      난이도      건수     총 시간
   --------------------------------------------------------------------
   Task 1c: 포트폴리오 직접 검토                5        4/5     50     250분
   Task 2:  면접 후보 수동 선정                30       3/5     1      30분
   Task 3a: 인터뷰 일정 조율                  20       2/5     1      20분
   Task 3b: 인터뷰 진행 (20명)               15       3/5     20     300분
   Task 4:  최종 합격자 수동 결정               60       5/5     1      60분
   Task 5a: 결과 이메일 수동 발송               2        1/5     50     100분
   Task 5b: 멘토 직접 매칭                   5        2/5     10     50분

   ▶ 총 소요 시간: 810분 (13.5시간)
   ▶ 인지 부하 합계: 340점
   ▶ 인간 담당 태스크 수: 7개

📌 Alt 3 (AI 최대 도입)
   AI: 대부분 | 인간: 핵심 의사결정 검토·승인만

   태스크                                 분/건      난이도      건수     총 시간
   --------------------------------------------------------------------
   Task 2:  AI 추천 검토 & 승인              15       2/5   

---

## 5. 종합 평가 결과 대시보드

3가지 평가 방법에서 측정한 결과를 통합하여 에이전트 시스템 전체를 평가합니다.

In [10]:
# ── 종합 평가 대시보드 ──────────────────────────────────────────────────────
print('\n' + '='*70)
print('📊 APD_alter3 동아리 모집 파이프라인 — 종합 평가 결과')
print('='*70)

print('\n┌─ 1. 효과성 (Effectiveness) ──────────────────────────────────────────┐')
print(f'│  [파일럿 테스트]  Eligibility Checker 정확도: {accuracy*100:.0f}%               │')
print(f'│  [LLM Judge]     Essay Analyzer 출력 품질:  {avg_essay_quality:.1f}/15점        │')
print(f'│  [LLM Judge]     Notification 이메일 품질:  {avg_email_quality:.1f}/20점        │')
print('└──────────────────────────────────────────────────────────────────────┘')

print('\n┌─ 2. 효율성 (Efficiency) ─────────────────────────────────────────────┐')
print(f'│  [파일럿 테스트]  평균 응답시간: {avg_rt:.2f}초/건                      │')
print(f'│  [시뮬레이션]     예상 처리량:   {60/avg_seq:.1f}건/분                   │')
print(f'│  [시뮬레이션]     100명 처리 비용: ${cost_per*100:.5f} USD           │')
print('└──────────────────────────────────────────────────────────────────────┘')

print('\n┌─ 3. 인간 작업부하 (Human Workload) ──────────────────────────────────┐')
print(f'│  [시뮬레이션]  Alt1 총 인간 작업: {t1}분 ({t1/60:.1f}시간)              │')
print(f'│  [시뮬레이션]  Alt3 총 인간 작업: {t3}분 ({t3/60:.1f}시간)              │')
print(f'│  AI 도입(Alt3)으로 인간 작업부하 {(1-t3/t1)*100:.0f}% 감소 ↓                  │')
print('└──────────────────────────────────────────────────────────────────────┘')

print('\n📋 에이전트별 종합 평가:')
print(f'  Eligibility Checker — 정확도: {accuracy*100:.0f}% | 속도: {avg_rt:.2f}초/건')
print(f'  Essay Analyzer      — 품질:  {avg_essay_quality:.1f}/15점')
print(f'  Notification Sender — 품질:  {avg_email_quality:.1f}/20점')
print('\n✅ 평가 완료.')


📊 APD_alter3 동아리 모집 파이프라인 — 종합 평가 결과

┌─ 1. 효과성 (Effectiveness) ──────────────────────────────────────────┐
│  [파일럿 테스트]  Eligibility Checker 정확도: 100%               │
│  [LLM Judge]     Essay Analyzer 출력 품질:  14.0/15점        │
│  [LLM Judge]     Notification 이메일 품질:  18.0/20점        │
└──────────────────────────────────────────────────────────────────────┘

┌─ 2. 효율성 (Efficiency) ─────────────────────────────────────────────┐
│  [파일럿 테스트]  평균 응답시간: 0.79초/건                      │
│  [시뮬레이션]     예상 처리량:   86.0건/분                   │
│  [시뮬레이션]     100명 처리 비용: $0.00570 USD           │
└──────────────────────────────────────────────────────────────────────┘

┌─ 3. 인간 작업부하 (Human Workload) ──────────────────────────────────┐
│  [시뮬레이션]  Alt1 총 인간 작업: 810분 (13.5시간)              │
│  [시뮬레이션]  Alt3 총 인간 작업: 345분 (5.8시간)              │
│  AI 도입(Alt3)으로 인간 작업부하 57% 감소 ↓                  │
└──────────────────────────────────────────────────────────────────────┘

📋 에이전트별 종합 평가:
  Eligibility Che

---

## 6. 실습 과제

### 과제 1: 테스트 케이스 추가 (파일럿 테스트)

아래 TODO 셀에서 `ELIGIBILITY_TEST_CASES`에 **에지 케이스 2개 이상**을 추가하고
정확도 변화를 확인하세요.

예시 에지 케이스:
- 학년 정보가 누락된 지원서
- 전공명이 모호한 경우 (예: "공학부" — 공과대학인지 불분명)

### 과제 2: 평가 루브릭 수정 (전문가 휴리스틱 평가)

`JUDGE_EMAIL_PROMPT`에 아래 기준을 **추가**하고 판정 결과가 어떻게 달라지는지 비교하세요.
- "오탈자 없음" (1점)
- "수신자 이름이 이메일 본문에 명시" (1점)

### 과제 3: 새 Alternative 작업부하 추정 (선택)

`APD_ALT` 딕셔너리에 **Alt 2** (중간 수준 AI 도입)를 추가하고
Alt 1 / Alt 2 / Alt 3을 한 번에 비교해보세요.

In [11]:
# ====================================================
# 과제 1: 테스트 케이스 추가
# ====================================================
MY_TEST_CASES = ELIGIBILITY_TEST_CASES.copy()

# TODO: 아래에 새 EvalCase를 추가하세요 (최소 2개)
# MY_TEST_CASES.append(EvalCase(
#     test_id='EC006',
#     input_text='이름: ???\n학년: ???학년\n전공: ???\n이수 과목: ???',
#     expected='PASS 또는 FAIL',
#     description='에지 케이스 설명'
# ))

checker2 = EligibilityCheckerAgent()
my_results = []
for tc in MY_TEST_CASES:
    r = checker2.run(tc.input_text)
    u = r.upper()
    actual = 'PASS' if ('PASS' in u and 'FAIL' not in u) else 'FAIL'
    my_results.append(actual == tc.expected)

my_acc = sum(my_results) / len(my_results)
print(f'📊 수정 후 정확도: {my_acc*100:.0f}% ({sum(my_results)}/{len(my_results)})')

# ====================================================
# 과제 2: 평가 루브릭 수정
# ====================================================
# TODO: 아래 JUDGE_EMAIL_PROMPT를 복사하여 "오탈자 없음", "수신자 이름 명시" 기준 추가

MY_JUDGE_EMAIL_PROMPT = JUDGE_EMAIL_PROMPT  # 수정 전 기준 (복사 후 수정하세요)
# my_email_judge_config = types.GenerateContentConfig(
#                                        system_instruction=MY_JUDGE_EMAIL_PROMPT)
# 수정 후 재실행 → 점수 변화 관찰

# ====================================================
# 과제 3 (선택): Alt 2 추가
# ====================================================
# TODO: APD_ALT에 'Alt 2 (중간 수준)' 항목을 추가하고
#       Alt 1 / Alt 2 / Alt 3을 한 번에 비교하는 루프를 실행하세요.

print('\n📋 실습 체크리스트:')
print('  [ ] 과제 1: 에지 케이스 테스트 케이스 2개 이상 추가')
print('  [ ] 과제 2: 이메일 평가 루브릭 확장 및 재실행')
print('  [ ] 과제 3 (선택): Alt 2 작업부하 추정 및 3-way 비교')

📊 수정 후 정확도: 100% (5/5)

📋 실습 체크리스트:
  [ ] 과제 1: 에지 케이스 테스트 케이스 2개 이상 추가
  [ ] 과제 2: 이메일 평가 루브릭 확장 및 재실행
  [ ] 과제 3 (선택): Alt 2 작업부하 추정 및 3-way 비교


---

## 7. 핵심 정리

| 개념 | 핵심 내용 |
|------|----------|
| **효과성 (Effectiveness)** | 정답과 비교한 정확도(%) 또는 루브릭 기준 품질 점수 |
| **효율성 (Efficiency)** | 응답 시간, 처리량(건/분), API 비용 |
| **인간 작업부하** | 소요 시간 × 건수, 인지 난이도 — AI 도입 전후 비교 |
| **파일럿 테스트** | 소규모 정답 데이터로 실제 실행, 자동 채점 |
| **시뮬레이션** | 대규모 시나리오 생성, 외삽으로 실운영 성능 추정 |
| **전문가 휴리스틱 평가** | LLM-as-Judge가 루브릭 기준으로 출력 품질 채점 |
| **LLM-as-Judge** | 정량 정답이 없는 출력(에세이, 이메일)의 품질 평가에 유용 |
| **APD Alternative 비교** | 평가 결과를 근거로 어떤 Alternative가 더 나은지 결정 |

---

```
exercise 1-1: GCP 환경 설정
 ↓
exercise 1-2: 에이전트 기초 — Eligibility Checker (시스템 프롬프트, Multi-turn, ReAct)
 ↓
exercise 2: 도구 & 메모리 — Application Collector / Essay Analyzer / Portfolio Evaluator
 ↓
exercise 3: 에이전트 아키텍처 — 반사형 / ReAct / 계획형 / 성찰형 + 멀티 에이전트 파이프라인
 ↓
exercise 4: 에이전트 평가 — 파일럿 테스트 / 시뮬레이션 / 전문가 휴리스틱 / 인간 작업부하
```

**4 exercises 과정 완료! 이제 APD 프레임워크로 에이전트 시스템을 설계하고 평가할 수 있습니다.**